In [16]:
import os, sys
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/wdi_solow_tidy.csv")

# sort is VERY important for time shifting
df = df.sort_values(["country", "year"]).reset_index(drop=True)

df.head()

# GDP per capita 10 years later
df["gdp_pc_t10"] = df.groupby("country")["gdp_per_capita"].shift(-1)

# 10-year average annual growth
df["growth_10y_avg"] = (
    np.log(df["gdp_pc_t10"]) - np.log(df["gdp_per_capita"])
) / 10

df[["country", "year", "gdp_per_capita", "gdp_pc_t10", "growth_10y_avg"]].head(10)

g = 0.02
delta = 0.03

# build n + g + δ
df["n_g_delta"] = df["population_growth"] / 100 + g + delta

# remove invalid values BEFORE log
df = df[
    (df["gdp_per_capita"] > 0) &
    (df["investment_rate"] > 0) &
    (df["n_g_delta"] > 0)
].copy()

# log features
df["ln_y0"] = np.log(df["gdp_per_capita"])
df["ln_s"] = np.log(df["investment_rate"] / 100)
df["ln_ngd"] = np.log(df["n_g_delta"])

ml_df = df.dropna(subset=[
    "growth_10y_avg",
    "ln_y0",
    "ln_s",
    "ln_ngd"
]).copy()

ml_df[["country", "year", "growth_10y_avg", "ln_y0", "ln_s", "ln_ngd"]].head()

features_theory = ["ln_y0", "ln_s", "ln_ngd"]
target = "growth_10y_avg"

print("Dataset shape:", ml_df.shape)
print("\nYears distribution:")
print(ml_df["year"].value_counts().sort_index())

ml_df.groupby("country")["year"].count().describe()

train_df = ml_df[ml_df["year"].isin([1990, 2000])].copy()
test_df  = ml_df[ml_df["year"] == 2010].copy()

print("Train years:")
print(train_df["year"].value_counts().sort_index())

print("\nTest years:")
print(test_df["year"].value_counts().sort_index())

features = ["ln_y0", "ln_s", "ln_ngd"]

X_train = train_df[features]
y_train = train_df["growth_10y_avg"]

X_test = test_df[features]
y_test = test_df["growth_10y_avg"]

print(X_train.shape, X_test.shape)

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor
import numpy as np

def evaluate(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    return {
        "model": name,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_test, preds),
        "R2_oos": r2_score(y_test, preds)
    }

results = []

# Mean baseline
results.append(evaluate(
    DummyRegressor(strategy="mean"),
    X_train, y_train, X_test, y_test,
    "Mean"
))

# Linear regression
results.append(evaluate(
    LinearRegression(),
    X_train, y_train, X_test, y_test,
    "Linear"
))

# Ridge
results.append(evaluate(
    Ridge(alpha=1.0),
    X_train, y_train, X_test, y_test,
    "Ridge"
))

# Lasso
results.append(evaluate(
    Lasso(alpha=0.001),
    X_train, y_train, X_test, y_test,
    "Lasso"
))

import pandas as pd
results_df = pd.DataFrame(results).sort_values("RMSE")

results_df


# ====================================
import os, sys
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/wdi_solow_raw_extended.csv")

# wide -> long
df_long = df.melt(
    id_vars=["country", "series"],
    var_name="year",
    value_name="value"
)

df_long["year"] = df_long["year"].str.replace("YR", "", regex=False).astype(int)

indicator_map = {
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "NE.GDI.TOTL.ZS": "investment_rate",
    "SP.POP.GROW": "population_growth",
    "SE.SEC.ENRR": "school_enrollment"
}

df_long["indicator"] = df_long["series"].map(indicator_map)

df_clean = df_long.pivot_table(
    index=["country", "year"],
    columns="indicator",
    values="value"
).reset_index()

df_clean.columns.name = None

df_clean.to_csv("../data/processed/wdi_solow_tidy_extended.csv", index=False)

df_clean.head()

df = pd.read_csv("../data/processed/wdi_solow_tidy_extended.csv")
df = df.sort_values(["country", "year"]).reset_index(drop=True)

# 10-year ahead GDP per capita
df["gdp_pc_t10"] = df.groupby("country")["gdp_per_capita"].shift(-1)

# average annual forward growth over next 10 years
df["growth_10y_avg"] = (
    np.log(df["gdp_pc_t10"]) - np.log(df["gdp_per_capita"])
) / 10

g = 0.02
delta = 0.03

df["n_g_delta"] = df["population_growth"] / 100 + g + delta

df = df[
    (df["gdp_per_capita"] > 0) &
    (df["investment_rate"] > 0) &
    (df["school_enrollment"] > 0) &
    (df["n_g_delta"] > 0)
].copy()

df["ln_y0"] = np.log(df["gdp_per_capita"])
df["ln_s"] = np.log(df["investment_rate"] / 100)
df["ln_ngd"] = np.log(df["n_g_delta"])
df["ln_h"] = np.log(df["school_enrollment"])

ml_df_h = df.dropna(subset=[
    "growth_10y_avg", "ln_y0", "ln_s", "ln_ngd", "ln_h"
]).copy()

ml_df_h[["country", "year", "growth_10y_avg", "ln_y0", "ln_s", "ln_ngd", "ln_h"]].head()

print("Dataset shape:", ml_df_h.shape)
print("\nYears distribution:")
print(ml_df_h["year"].value_counts().sort_index())

train_df_h = ml_df_h[ml_df_h["year"].isin([1990, 2000])].copy()
test_df_h  = ml_df_h[ml_df_h["year"] == 2010].copy()

print("Train years:")
print(train_df_h["year"].value_counts().sort_index())

print("\nTest years:")
print(test_df_h["year"].value_counts().sort_index())

features_h = ["ln_y0", "ln_s", "ln_ngd", "ln_h"]
target = "growth_10y_avg"

X_train_h = train_df_h[features_h]
y_train_h = train_df_h[target]

X_test_h = test_df_h[features_h]
y_test_h = test_df_h[target]

print(X_train_h.shape, X_test_h.shape)

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

def evaluate(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    return {
        "model": name,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_test, preds),
        "R2_oos": r2_score(y_test, preds)
    }

results_h = []

results_h.append(evaluate(
    DummyRegressor(strategy="mean"),
    X_train_h, y_train_h, X_test_h, y_test_h,
    "Mean"
))

results_h.append(evaluate(
    LinearRegression(),
    X_train_h, y_train_h, X_test_h, y_test_h,
    "Linear + H"
))

results_h.append(evaluate(
    Ridge(alpha=1.0),
    X_train_h, y_train_h, X_test_h, y_test_h,
    "Ridge + H"
))

results_h.append(evaluate(
    Lasso(alpha=0.001),
    X_train_h, y_train_h, X_test_h, y_test_h,
    "Lasso + H"
))

results_h_df = pd.DataFrame(results_h).sort_values("RMSE")
results_h_df

results_df["feature_set"] = "theory_only"
results_h_df["feature_set"] = "theory_plus_h"

comparison_df = pd.concat([results_df, results_h_df], ignore_index=True)
comparison_df.sort_values(["feature_set", "RMSE"])

ml_df_h.to_csv("../data/processed/solow_ml_panel_extended.csv", index=False)
comparison_df.to_csv("../reports/tables/model_comparison_extended.csv", index=False)

comparison_df = pd.concat([results_df, results_h_df], ignore_index=True)
comparison_df.sort_values(["feature_set", "RMSE"])

Dataset shape: (571, 11)

Years distribution:
year
1990    166
2000    196
2010    209
Name: count, dtype: int64
Train years:
year
1990    166
2000    196
Name: count, dtype: int64

Test years:
year
2010    209
Name: count, dtype: int64
(362, 3) (209, 3)
Dataset shape: (376, 13)

Years distribution:
year
1990     67
2000    146
2010    163
Name: count, dtype: int64
Train years:
year
1990     67
2000    146
Name: count, dtype: int64

Test years:
year
2010    163
Name: count, dtype: int64
(213, 4) (163, 4)


,model,RMSE,MAE,R2_oos,feature_set
0,Ridge,0.023648,0.016748,-0.037848,theory_only
1,Linear,0.023672,0.016789,-0.039994,theory_only
2,Lasso,0.024428,0.017832,-0.107477,theory_only
3,Mean,0.025196,0.018678,-0.178204,theory_only
4,Ridge + H,0.023287,0.016276,-0.076657,theory_plus_h
5,Linear + H,0.023371,0.016450,-0.084494,theory_plus_h
6,Lasso + H,0.023553,0.016601,-0.101385,theory_plus_h
7,Mean,0.024751,0.018626,-0.216297,theory_plus_h
